# 03c — PCA Applied

## The One-Sentence Version
PCA on real data: digits, patient vitals, and the explained variance curve.

## What You'll Build Intuition For
- How PCA performs on real datasets (digits, patient records)
- How to choose the number of components (elbow, threshold, Kaiser)
- What principal component "images" look like and what they mean
- When PCA fails and why

## Prerequisites
- [03b — PCA From Scratch](03b_pca_from_scratch.ipynb) — the algorithm step by step

## The Story

In 03a we understood the geometry of projection. In 03b we built PCA from scratch. Now we apply it to real data and face the practical decisions every practitioner has to make.

The biggest question: **how many components?** Keep too few and you lose important structure. Keep too many and you're dragging noise along for the ride. There's no single right answer, but there are solid heuristics — and they usually agree.

We'll also see PCA's limit: it can only find linear structure. When the data is curved — like a Swiss roll — PCA flattens it instead of unrolling it. That failure motivates everything in Module 04.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_digit_data, make_patient_data, make_swiss_roll_data

apply_style()
rng = np.random.default_rng(42)

## Digits Dataset

Each handwritten digit is an 8×8 image — a point in 64-dimensional space. PCA finds the directions of maximum variation across all digit images.

In [ ]:
# Load digits and fit full PCA
data, target, images = make_digit_data()
pca_full = PCA().fit(data)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_95 = np.searchsorted(cumvar, 0.95) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Per-component variance
ax1.bar(range(1, 21), pca_full.explained_variance_ratio_[:20],
        color=COLOURS["signal"], alpha=0.8, edgecolor="white")
ax1.set_xlabel("Component")
ax1.set_ylabel("Variance Explained")
ax1.set_title("Per Component (first 20)")

# Cumulative
ax2.plot(range(1, 65), cumvar, "o-", color=COLOURS["signal"], markersize=3, linewidth=2)
ax2.axhline(y=0.95, color=COLOURS["accent"], linestyle="--", alpha=0.7, label="95%")
ax2.axvline(n_95, color=COLOURS["accent"], linewidth=0.8, alpha=0.4)
ax2.annotate(f"{n_95} components", (n_95, 0.95),
             textcoords="offset points", xytext=(10, -15), fontsize=10,
             arrowprops=dict(arrowstyle="->", color=COLOURS["accent"]))
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance Explained")
ax2.set_title("Cumulative")
ax2.legend()
ax2.set_ylim(0, 1.05)

fig.suptitle(f"Digits: 64D → {n_95} components for 95% variance", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03c_digits_variance.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"{n_95} of 64 components capture 95% of variance.")
print(f"The first 2 alone capture {cumvar[1]:.1%}.")

In [ ]:
# 2D projection colour-coded by digit
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(data)

fig, ax = plt.subplots(figsize=(10, 8))
for digit in range(10):
    mask = target == digit
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=12, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax.set_xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%})")
ax.set_title("Digits in 2D PCA space — clusters emerge")
ax.legend(title="Digit", markerscale=3, fontsize=9)
plt.tight_layout()
plt.savefig("visuals/03c_digits_2d.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Principal component "images" — eigendigits
fig, axes = plt.subplots(1, 6, figsize=(14, 2.5))

# Mean digit
axes[0].imshow(pca_full.mean_.reshape(8, 8), cmap="gray_r")
axes[0].set_title("Mean", fontsize=10)
axes[0].set_xticks([])
axes[0].set_yticks([])

# First 5 principal components as images
for i in range(5):
    axes[i+1].imshow(pca_full.components_[i].reshape(8, 8), cmap="RdBu_r")
    axes[i+1].set_title(f"PC{i+1} ({pca_full.explained_variance_ratio_[i]:.1%})", fontsize=10)
    axes[i+1].set_xticks([])
    axes[i+1].set_yticks([])

fig.suptitle("Eigendigits: what each principal component looks like", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03c_eigendigits.png", dpi=150, bbox_inches="tight")
plt.show()

print("Each PC image shows one axis of variation across all digits.")
print("PC1 captures gross brightness. Later PCs capture finer details.")
print("Any digit can be approximated as: mean + c1·PC1 + c2·PC2 + ...")

Each principal component is an 8×8 "image" that shows one direction of variation. The actual digit images are linear combinations of these component images, weighted by their PC scores. More components = more detail in the reconstruction.

## Patient Vitals

Synthetic patient data with 5 signal dimensions embedded in 30 noise dimensions. PCA should cleanly separate signal from noise.

In [ ]:
# Patient data: 5 signal + 30 noise
X_patients, feature_names, signal_mask = make_patient_data(
    n=500, d_signal=5, d_noise=30, seed=42
)

# Standardise (important when features have different scales)
scaler = StandardScaler()
X_patients_scaled = scaler.fit_transform(X_patients)

pca_patients = PCA().fit(X_patients_scaled)
cumvar_patients = np.cumsum(pca_patients.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Scree plot
n_show = 20
colours_bar = [COLOURS["signal"] if ev > 1.0 else COLOURS["neutral"]
               for ev in pca_patients.explained_variance_[:n_show]]
ax1.bar(range(1, n_show+1), pca_patients.explained_variance_[:n_show],
        color=colours_bar, alpha=0.8, edgecolor="white")
ax1.axhline(y=1.0, color=COLOURS["accent"], linestyle="--", alpha=0.7,
            label="Kaiser criterion (λ=1)")
ax1.set_xlabel("Component")
ax1.set_ylabel("Eigenvalue")
ax1.set_title("Scree Plot")
ax1.legend()

# Cumulative
ax2.plot(range(1, 36), cumvar_patients, "o-", color=COLOURS["signal"],
         markersize=3, linewidth=2)
ax2.axhline(y=0.95, color=COLOURS["accent"], linestyle="--", alpha=0.7, label="95%")
n_95_patients = np.searchsorted(cumvar_patients, 0.95) + 1
ax2.axvline(n_95_patients, color=COLOURS["accent"], linewidth=0.8, alpha=0.4)
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance Explained")
ax2.set_title("Cumulative")
ax2.legend()
ax2.set_ylim(0, 1.05)

fig.suptitle(f"Patient data: 35 features, {n_95_patients} components for 95%",
             fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03c_patients_variance.png", dpi=150, bbox_inches="tight")
plt.show()

n_kaiser = np.sum(pca_patients.explained_variance_ > 1.0)
print(f"True signal dimensions: {signal_mask.sum()}")
print(f"Kaiser criterion (eigenvalue > 1): {n_kaiser} components")
print(f"95% variance threshold: {n_95_patients} components")
print(f"PCA separates signal from noise.")

## Choosing k: The Elbow and the Threshold

Three common methods for choosing the number of components:

1. **95% explained variance** — keep enough components to explain 95% of variance
2. **Kaiser criterion** — keep components with eigenvalue > 1 (when data is standardised)
3. **Scree plot elbow** — look for the bend in the eigenvalue curve

They usually agree. When they don't, that's a sign the data doesn't have a clean low-rank structure.

In [ ]:
# Compare all three methods on the patient data
eigenvalues_patients = pca_patients.explained_variance_

# Method 1: 95% variance
n_95 = np.searchsorted(cumvar_patients, 0.95) + 1

# Method 2: Kaiser criterion
n_kaiser = np.sum(eigenvalues_patients > 1.0)

# Method 3: Elbow via maximum second derivative
diffs = np.diff(eigenvalues_patients[:15])
diffs2 = np.diff(diffs)
n_elbow = np.argmax(np.abs(diffs2)) + 2  # +2 for offset from double diff

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, 16), eigenvalues_patients[:15], "o-",
        color=COLOURS["signal"], markersize=6, linewidth=2)

ax.axvline(n_95, color=COLOURS["accent"], linewidth=2, linestyle="--",
           label=f"95% variance: k={n_95}")
ax.axvline(n_kaiser, color=COLOURS["success"], linewidth=2, linestyle="-.",
           label=f"Kaiser (λ>1): k={n_kaiser}")
ax.axvline(n_elbow, color=COLOURS["noise"], linewidth=2, linestyle=":",
           label=f"Elbow: k={n_elbow}")
ax.axhline(y=1.0, color=COLOURS["neutral"], linewidth=0.8, alpha=0.4)

ax.set_xlabel("Component")
ax.set_ylabel("Eigenvalue")
ax.set_title("Three methods for choosing k (usually agree)")
ax.legend()
plt.tight_layout()
plt.savefig("visuals/03c_choosing_k.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"True signal dimension: {signal_mask.sum()}")
print(f"95% variance threshold: {n_95}")
print(f"Kaiser criterion:       {n_kaiser}")
print(f"Elbow detection:        {n_elbow}")
print(f"\nAll three point to roughly the same answer.")

> **Practical advice:** Start with the 95% threshold as your default. If interpretability matters, look at the scree plot and pick the elbow — components beyond the elbow contribute diminishing returns. Kaiser criterion is most meaningful when data is standardised.

## When PCA Fails

PCA assumes the important structure is **linear** — that the data varies along straight lines through the space. When the data lies on a **curved** surface, PCA can't unroll it.

The Swiss roll is the classic demonstration.

In [ ]:
# Swiss roll: a 2D surface rolled up in 3D
X_swiss, colour = make_swiss_roll_data(n=1500, noise=0.3, seed=42)

# PCA projection
pca_swiss = PCA(n_components=2)
X_swiss_pca = pca_swiss.fit_transform(X_swiss)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# PCA projection — fails to unroll
ax1.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], s=5, alpha=0.5,
            c=colour, cmap="viridis")
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")
ax1.set_title("PCA: crushes the roll flat")

# What we want: colour-preserving unrolling
ax2.scatter(colour, X_swiss[:, 1], s=5, alpha=0.5, c=colour, cmap="viridis")
ax2.set_xlabel("Position along manifold")
ax2.set_ylabel("Height")
ax2.set_title("Ideal: unrolled manifold coordinates")

fig.suptitle("PCA can't unroll a curved manifold", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03c_pca_fails_swiss.png", dpi=150, bbox_inches="tight")
plt.show()

print("Left: PCA projects the Swiss roll — colours overlap badly.")
print("Nearby colours should stay nearby. PCA mixes them because it can't see curves.")
print("\nThis is why we need Module 04: nonlinear methods (t-SNE, UMAP, Isomap).")

In [ ]:
# Explained variance for the Swiss roll — PCA "thinks" it needs 2 components
pca_swiss_full = PCA().fit(X_swiss)

print("Explained variance per component:")
for i, ev in enumerate(pca_swiss_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {ev:.1%}")

print(f"\nPCA says 2 components capture {sum(pca_swiss_full.explained_variance_ratio_[:2]):.1%}.")
print("Technically true — but the 2D projection scrambles the structure.")
print("High explained variance ≠ good representation when the manifold is curved.")

> **Key insight:** PCA's explained variance metric can be misleading on curved data. A projection can capture 95% of variance while completely destroying the neighbourhood structure. When your data has nonlinear manifold structure, you need nonlinear methods.

<details>
<summary><b>The Maths Behind This</b></summary>

PCA finds the linear subspace that minimises reconstruction error:

$$\min_W \sum_i \|\mathbf{x}_i - WW^T\mathbf{x}_i\|^2$$

where $W \in \mathbb{R}^{d \times k}$ has orthonormal columns. This is optimal *if* the data lies near a flat hyperplane.

For the Swiss roll, the data lies on a 2D manifold that is intrinsically flat but **extrinsically curved** in 3D. PCA projects onto the best-fitting plane, which cuts through the rolled surface and merges points that are far apart along the manifold.

The fix: methods that measure **geodesic distance** (distance along the manifold) instead of Euclidean distance. Isomap, t-SNE, and UMAP all do this in different ways.

</details>

## Where This Matters

**Quality control:** In manufacturing, sensors produce hundreds of measurements per part. PCA to 2-3 components creates a "quality space" where normal parts cluster tightly. A part that's far from the cluster in PC-space has an unusual pattern — even if no single measurement is out of range.

**Genomics preprocessing:** Gene expression datasets have ~20,000 features. Running PCA as a first step reveals batch effects (samples cluster by processing date, not biology) and identifies the effective dimensionality before downstream analysis.

## Feynman Check

1. **How many PCA components do you need for the digits dataset? Why?**

2. **What does a PCA eigenface (principal component image) look like and what does it mean?**

You've now seen PCA succeed and fail. For the full picture of linear methods, let's look at PCA's cousins in **03d — SVD and Friends**.